# Model Training — Smart Premium Prediction
Continues from `preprocessing.ipynb`.  
Trains multiple regressors with **RandomizedSearchCV** + **MLflow** tracking, then saves the best model.

## 1. Imports

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load Dataset 
train_df = pd.read_csv(r"...\data\playground-series-s4e12\train.csv")
train_df

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,...,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,...,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,...,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,...,1.0,14.0,NaN,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,...,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,...,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1199995,1199995,36.0,Female,27316.0,Married,0.0,Master's,Unemployed,13.772907,Urban,...,NaN,5.0,372.0,3.0,2023-05-03 15:21:39.257696,Poor,No,Daily,Apartment,1303.0
1199996,1199996,54.0,Male,35786.0,Divorced,NaN,Master's,Self-Employed,11.483482,Rural,...,NaN,10.0,597.0,4.0,2022-09-10 15:21:39.134960,Poor,No,Weekly,Apartment,821.0
1199997,1199997,19.0,Male,51884.0,Divorced,0.0,Master's,NaN,14.724469,Suburban,...,0.0,19.0,NaN,6.0,2021-05-25 15:21:39.106582,Good,No,Monthly,Condo,371.0
1199998,1199998,55.0,Male,NaN,Single,1.0,PhD,NaN,18.547381,Suburban,...,1.0,7.0,407.0,4.0,2021-09-19 15:21:39.190215,Poor,No,Daily,Apartment,596.0


In [3]:
print(train_df.shape)

(1200000, 21)


In [4]:
# Drop unwanted columns
train_df.drop(["id", "Customer Feedback"], axis=1, inplace=True)

# Convert to datetime
train_df["Policy Start Date"] = pd.to_datetime(train_df["Policy Start Date"], errors="coerce")

# Extract date features
train_df["Policy_Year"] = train_df["Policy Start Date"].dt.year
train_df["Policy_Month"] = train_df["Policy Start Date"].dt.month
train_df["Policy_Day"] = train_df["Policy Start Date"].dt.day

# Remove original date column
train_df.drop("Policy Start Date", axis=1, inplace=True)

# Check result
train_df.head()

,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,...,Vehicle Age,Credit Score,Insurance Duration,Smoking Status,Exercise Frequency,Property Type,Premium Amount,Policy_Year,Policy_Month,Policy_Day
0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,Premium,...,17.0,372.0,5.0,No,Weekly,House,2869.0,2023,12,23
1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,Comprehensive,...,12.0,694.0,2.0,Yes,Monthly,House,1483.0,2023,6,12
2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,Premium,...,14.0,NaN,3.0,Yes,Weekly,House,567.0,2023,9,30
3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,Basic,...,0.0,367.0,1.0,Yes,Daily,Apartment,765.0,2024,6,12
4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,Premium,...,8.0,598.0,4.0,Yes,Weekly,House,2022.0,2021,12,1


In [5]:
# Target Variable
y = train_df["Premium Amount"]

# Features
X = train_df.drop("Premium Amount", axis=1)

print("Features Shape :", X.shape)
print("Target Shape :", y.shape)

Features Shape : (1200000, 20)
Target Shape : (1200000,)


In [6]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train Shape :", X_train.shape)
print("y_train Shape :", y_train.shape)

X_train Shape : (960000, 20)
y_train Shape : (960000,)


In [7]:
preprocessor = joblib.load("models/preprocessor.pkl")

print("Preprocessor Loaded Successfully")

Preprocessor Loaded Successfully


In [8]:
print(type(preprocessor))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [9]:
def evaluate_model(model_name, y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name}")
    print("-" * 30)
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R2   : {r2:.4f}")

    return rmse, mae, r2

In [10]:
models = {

    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },

    "DecisionTree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "model__max_depth": [5, 10, 15, None],
            "model__min_samples_split": [2, 5, 10]
        }
    },

    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "model__n_estimators": [50],
            "model__max_depth": [6],
            "model__min_samples_split": [2]
        }
    },

    "XGBoost": {
        "model": XGBRegressor(
            random_state=42,
            objective="reg:squarederror"
        ),
        "params": {
            "model__n_estimators": [50],
            "model__max_depth": [6],
            "model__learning_rate": [0.05]
        }
    }
}

In [11]:
models

{'LinearRegression': {'model': LinearRegression(), 'params': {}},
 'DecisionTree': {'model': DecisionTreeRegressor(random_state=42),
  'params': {'model__max_depth': [5, 10, 15, None],
   'model__min_samples_split': [2, 5, 10]}},
 'RandomForest': {'model': RandomForestRegressor(random_state=42),
  'params': {'model__n_estimators': [50],
   'model__max_depth': [6],
   'model__min_samples_split': [2]}},
 'XGBoost': {'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          

In [12]:
best_model      = None
best_rmse       = float("inf")
best_model_name = None

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("SmartPremium_Experiment")

for model_name, config in models.items():

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    with mlflow.start_run(run_name=model_name):

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", config["model"])
        ])

        # Hyperparameter Tuning
        if config["params"]:
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=config["params"],
                n_iter=5,
                cv=3,
                scoring="neg_root_mean_squared_error",
                random_state=42,
                n_jobs=-1
            )
            search.fit(X_train, y_train)
            final_model = search.best_estimator_
            mlflow.log_params(search.best_params_)
        else:
            final_model = pipeline
            final_model.fit(X_train, y_train)

        # Prediction
        y_pred = final_model.predict(X_test)

        rmse, mae, r2 = evaluate_model(model_name, y_test, y_pred)

        # MLflow Logging
        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAE",  mae)
        mlflow.log_metric("R2",   r2)

        # Model Logging    
        mlflow.sklearn.log_model(
            sk_model=final_model,
            name="model"
        )

        # Best Model Selection
        if rmse < best_rmse:
            best_rmse       = rmse
            best_model      = final_model
            best_model_name = model_name

print("\n" + "="*60)
print(f"Best Model : {best_model_name}")
print(f"Best RMSE  : {best_rmse:.4f}")
print("="*60)

2026/07/02 18:23:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/02 18:23:48 INFO mlflow.store.db.utils: Updating database tables
2026/07/02 18:23:50 INFO mlflow.tracking.fluent: Experiment with name 'SmartPremium_Experiment' does not exist. Creating a new experiment.



Training: LinearRegression

LinearRegression
------------------------------
RMSE : 863.2730
MAE  : 667.2839
R2   : 0.0027


2026/07/02 18:24:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Training: DecisionTree


2026/07/02 18:26:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



DecisionTree
------------------------------
RMSE : 846.9494
MAE  : 640.2733
R2   : 0.0401

Training: RandomForest


2026/07/02 18:38:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RandomForest
------------------------------
RMSE : 849.1279
MAE  : 648.0372
R2   : 0.0352

Training: XGBoost


2026/07/02 18:38:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



XGBoost
------------------------------
RMSE : 847.6912
MAE  : 649.5145
R2   : 0.0384

Best Model : DecisionTree
Best RMSE  : 846.9494


In [13]:
joblib.dump(best_model, "models/best_model.pkl", compress=3)
print(f"\nBest Model : {best_model_name}")
print(f"Best RMSE  : {best_rmse:.4f}")  
print("Best Model Saved Successfully")



Best Model : DecisionTree
Best RMSE  : 846.9494
Best Model Saved Successfully
